# 🏗️ Day 02a: nn.Module — Build Real Networks

---

## 🎯 What You'll Master Today
- nn.Module class structure
- nn.Linear, nn.ReLU, nn.Sequential
- Loss functions & optimizers
- Building custom networks

**Goal:** Build any feed-forward network from memory using nn.Module.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  nn.Module — The Base Class of All PyTorch Models                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  class MyModel(nn.Module):                                       ║
║      def __init__(self):                                         ║
║          super().__init__()                                      ║
║          self.layers = ...       ← define layers here           ║
║                                                                   ║
║      def forward(self, x):                                       ║
║          return self.layers(x)   ← define computation here      ║
║                                                                   ║
║  model(x)    → calls forward(x) automatically                   ║
║  model.parameters() → all learnable params                      ║
║  model.train() / model.eval() → training/inference mode         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ============================================================
# 1. nn.Sequential — Quick & Simple
# ============================================================

# Binary classifier: 4 features → 1 output
model_seq = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# Test forward pass
x = torch.randn(5, 4)  # batch of 5, 4 features
output = model_seq(x)
print(f"Input:  {x.shape}")
print(f"Output: {output.shape}")
print(f"Predictions: {output.detach().flatten().numpy().round(4)}")

# Count parameters
total_params = sum(p.numel() for p in model_seq.parameters())
trainable = sum(p.numel() for p in model_seq.parameters() if p.requires_grad)
print(f"\nParameters: {total_params} total, {trainable} trainable")

In [ ]:
# ============================================================
# 2. Custom nn.Module — Full Control
# ============================================================

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()  # MUST call super().__init__()
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.fc3(x)  # raw logits (no sigmoid — use BCEWithLogitsLoss)
        return x

model = BinaryClassifier(input_dim=10, hidden_dim=32)
print(model)

# Forward pass
x = torch.randn(8, 10)  # batch=8, features=10
model.train()  # dropout active
logits_train = model(x)
model.eval()   # dropout disabled
with torch.no_grad():
    logits_eval = model(x)

print(f"\nTrain logits: {logits_train.detach().flatten()[:4].numpy().round(4)}")
print(f"Eval logits:  {logits_eval.flatten()[:4].numpy().round(4)}")
print("\n💡 train() vs eval() matters for Dropout and BatchNorm!")

In [ ]:
# ============================================================
# 3. Loss Functions — The Main Ones
# ============================================================

# Binary classification
logits = torch.tensor([2.0, -1.0, 0.5])  # raw scores (no sigmoid)
targets = torch.tensor([1.0, 0.0, 1.0])

bce_logits = nn.BCEWithLogitsLoss()  # sigmoid + BCE in one (more stable)
loss1 = bce_logits(logits, targets)
print(f"BCEWithLogitsLoss: {loss1.item():.4f}")

# Multi-class classification
logits_mc = torch.randn(3, 5)  # batch=3, classes=5 (raw logits)
targets_mc = torch.tensor([2, 0, 4])  # class indices (NOT one-hot!)

ce_loss = nn.CrossEntropyLoss()  # softmax + NLLLoss in one
loss2 = ce_loss(logits_mc, targets_mc)
print(f"CrossEntropyLoss: {loss2.item():.4f}")

# Regression
predictions = torch.tensor([2.5, 3.1, 4.8])
actuals = torch.tensor([2.0, 3.0, 5.0])

mse = nn.MSELoss()(predictions, actuals)
mae = nn.L1Loss()(predictions, actuals)
print(f"MSELoss: {mse.item():.4f}")
print(f"L1Loss:  {mae.item():.4f}")

print("\n💡 BCEWithLogitsLoss: pass raw logits (no sigmoid)")
print("   CrossEntropyLoss: pass raw logits (no softmax) + class indices")

In [ ]:
# ============================================================
# 4. Optimizers
# ============================================================

model = BinaryClassifier(input_dim=10)

# Adam — default choice
optimizer = optim.Adam(model.parameters(), lr=0.001)

# SGD with momentum
# optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# AdamW (decoupled weight decay — better for transformers)
# optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# The training step pattern
x = torch.randn(16, 10)
y = torch.randint(0, 2, (16, 1)).float()

model.train()
for step in range(5):
    # 1. Forward
    logits = model(x)
    loss = nn.BCEWithLogitsLoss()(logits, y)
    
    # 2. Backward
    optimizer.zero_grad()   # MUST zero before backward
    loss.backward()
    
    # 3. Update
    optimizer.step()
    
    print(f"Step {step}: Loss = {loss.item():.4f}")

print("\n💡 Pattern: zero_grad() → backward() → step()")

In [ ]:
# ============================================================
# 5. Multi-class Network — Complete Example
# ============================================================

class MultiClassNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),  # no softmax — CE does it
        )
    
    def forward(self, x):
        return self.net(x)

# Create model
model = MultiClassNet(input_dim=20, num_classes=5)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Fake data
X_train = torch.randn(200, 20)
y_train = torch.randint(0, 5, (200,))  # class indices

# Quick training
model.train()
for epoch in range(20):
    logits = model(X_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 5 == 0:
        preds = logits.argmax(dim=1)
        acc = (preds == y_train).float().mean()
        print(f"Epoch {epoch:2d} | Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

# Inference
model.eval()
with torch.no_grad():
    test_input = torch.randn(3, 20)
    probs = torch.softmax(model(test_input), dim=1)
    preds = probs.argmax(dim=1)
    print(f"\nPredictions: {preds.numpy()}")
    print(f"Confidence:  {probs.max(dim=1).values.numpy().round(4)}")

In [ ]:
# ============================================================
# 6. Save & Load Models
# ============================================================
import tempfile, os

save_path = os.path.join(tempfile.gettempdir(), 'model.pth')

# Save (state_dict = weights only, recommended)
torch.save(model.state_dict(), save_path)
print(f"Saved state_dict to {save_path}")
print(f"Size: {os.path.getsize(save_path) / 1024:.1f} KB")

# Load
model_loaded = MultiClassNet(input_dim=20, num_classes=5)
model_loaded.load_state_dict(torch.load(save_path, weights_only=True))
model_loaded.eval()

# Verify
with torch.no_grad():
    out_orig = model(test_input)
    out_loaded = model_loaded(test_input)
    match = torch.allclose(out_orig, out_loaded)
    print(f"\nOutputs match: {match}")

os.remove(save_path)

print("\n💡 Save: torch.save(model.state_dict(), path)")
print("   Load: model.load_state_dict(torch.load(path))")
print("   Always save state_dict, not the full model.")

---

## 🗺️ nn.Module Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  nn.Module — Building Blocks                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  LAYERS:                                                         ║
║    nn.Linear(in, out)     nn.Conv2d()   nn.LSTM()              ║
║    nn.BatchNorm1d()       nn.Dropout()  nn.Embedding()         ║
║                                                                   ║
║  ACTIVATIONS:                                                    ║
║    nn.ReLU()  nn.Sigmoid()  nn.Softmax(dim=1)                  ║
║    F.relu()   F.gelu()      (functional versions)               ║
║                                                                   ║
║  LOSSES:                                                         ║
║    Binary:  BCEWithLogitsLoss (logits, no sigmoid)              ║
║    Multi:   CrossEntropyLoss  (logits, class indices)           ║
║    Regress: MSELoss / L1Loss                                    ║
║                                                                   ║
║  TRAIN LOOP: zero_grad → forward → loss → backward → step     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is nn.Module? | Base class for all models. Define layers in __init__, computation in forward(). Tracks parameters |
| 2 | train() vs eval()? | train(): dropout active, BN uses batch stats. eval(): dropout off, BN uses running stats |
| 3 | Why BCEWithLogitsLoss over BCELoss? | Combines sigmoid + BCE in one step. More numerically stable (uses log-sum-exp trick) |
| 4 | How to save/load models? | Save state_dict (weights only). Load with load_state_dict. Never pickle full model |
| 5 | zero_grad() → backward() → step()? | Zero old gradients, compute new ones via backprop, update weights. This order is mandatory |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • nn.Module: __init__ (layers) + forward (logic)      │
## │  • Sequential for simple, custom class for complex      │
## │  • BCE: logits (no sigmoid). CE: logits + class idx     │
## │  • Loop: zero_grad → forward → loss → backward → step  │
## │  • Save state_dict, not full model                      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Training Loop** — DataLoader, epochs, validation, early stopping